# 03 — Grid Topology Analysis

Visualise the grid graph, analyse connectivity, find critical nodes.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pickle
import networkx as nx
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from uk_energy.config import GRAPH_PICKLE, OUTPUT_DIR

## 1. Load Graph

In [ ]:
if GRAPH_PICKLE.exists():
    with open(GRAPH_PICKLE, 'rb') as f:
        G = pickle.load(f)
    print(f'Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    print(f'Graph metadata: {G.graph.get("metadata", {})}')
else:
    print('Graph not found — run: python -m uk_energy build-graph')
    # Build from scratch
    from uk_energy.graph.builder import build_grid_graph
    G = build_grid_graph()
    print(f'Built graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 2. Node Type Distribution

In [ ]:
from collections import Counter
node_types = Counter(data.get('node_type', 'unknown') for _, data in G.nodes(data=True))
print('Node type distribution:')
for ntype, count in sorted(node_types.items()):
    print(f'  {ntype}: {count:,}')

## 3. Connectivity Analysis

In [ ]:
from uk_energy.graph.topology import analyse_connectivity, find_critical_nodes

connectivity = analyse_connectivity(G)
print('Connectivity analysis:')
for k, v in connectivity.items():
    print(f'  {k}: {v}')

## 4. Regional Capacity Summary

In [ ]:
from uk_energy.graph.topology import regional_capacity_summary, fuel_capacity_summary

regional = regional_capacity_summary(G)
print('Regional generation capacity:')
display(regional)

fuel_cap = fuel_capacity_summary(G)
print('\nFuel type capacity:')
display(fuel_cap)

## 5. Interconnector Analysis

In [ ]:
from uk_energy.graph.topology import interconnector_analysis

ic = interconnector_analysis(G)
print(f'Total interconnectors: {ic["total_interconnector_count"]}')
print(f'Total import capacity: {ic["total_import_capacity_mw"]:,} MW')
print('\nBy interconnector:')
for ic_id, info in ic['by_interconnector'].items():
    print(f'  {ic_id}: {info["capacity_mw"]:,} MW — {info["name"]}')
print('\nBy country:')
for country, cap in sorted(ic['by_country'].items(), key=lambda x: -x[1]):
    print(f'  {country}: {cap:,} MW')

## 6. Degree Distribution

In [ ]:
import plotly.express as px
import numpy as np

degrees = [d for _, d in G.degree()]
fig = px.histogram(
    x=degrees,
    nbins=50,
    title='Node Degree Distribution',
    labels={'x': 'Degree (connections)', 'y': 'Count'},
    log_y=True,
)
fig.show()
print(f'Mean degree: {np.mean(degrees):.2f}')
print(f'Max degree: {max(degrees)}')
print(f'Isolated nodes (degree=0): {sum(1 for d in degrees if d == 0)}')